# Praktikum Sub-CPMK-4 &mdash; Transformasi Ruang Warna untuk Analisis dan Pemrosesan Citra

**Pengolahan Citra Digital (IF2514505)** &nbsp;|&nbsp; Semester 5 &nbsp;|&nbsp; 1 SKS Praktikum &nbsp;|&nbsp; Durasi sasaran 135 menit (rentang 120&ndash;150)

**Program Studi Informatika &mdash; Institut Teknologi Kalimantan**

---

| | |
|---|---|
| **Nama** | ................................................ |
| **NIM** | ................................................ |
| **Kelas / Kelompok** | ................................................ |
| **Tanggal praktikum** | ................................................ |

---

### Capaian yang diukur
**Sub-CPMK-4**: Mahasiswa mampu menerapkan transformasi ruang warna citra untuk keperluan analisis dan pemrosesan citra.
Indikator: (1) ketepatan konversi antarruang warna dan pemilihan ruang warna yang sesuai persoalan; (2) ketepatan laporan praktikum sesuai kaidah penulisan ilmiah.

### Rujukan
Gonzalez, R.C. dan Woods, R.E. (2018) *Digital Image Processing*, edisi ke-4, Bab 6 (§6.1&ndash;6.8). Nomor persamaan mengikuti buku tersebut.
HSV tidak dibahas Gonzalez; definisinya mengikuti Smith (1978) dan dokumentasi OpenCV.

### Petunjuk pengerjaan
1. Jalankan sel berurutan. Sel bertanda `# TODO` wajib Anda lengkapi sendiri.
2. Setiap blok **Analisis** diisi kalimat Anda sendiri dan **wajib memuat angka** dari keluaran Anda.
3. Citra uji dibangkitkan di dalam notebook (benih tetap); citra `coffee` diambil dari `skimage.data`. **Tidak perlu mengunggah berkas.**
4. Semua citra di notebook ini berurutan kanal **RGB**. Untuk OpenCV pakai kode `cv2.COLOR_RGB2...`, bukan `BGR2...`.
5. Masker kebenaran (`gt_...`) hanya dipakai untuk **menilai** hasil, bukan untuk menentukan ambang.
6. Bagian **[Pengayaan]** boleh dilewati; jalur inti selesai dalam &plusmn;120 menit.
7. Kumpulkan dalam keadaan **sudah dijalankan penuh** dengan nama `PraktikumPCD_SubCPMK4_NIM_Nama.ipynb`, beserta laporan (Bagian 6).

| Bagian | Isi | Menit |
|---|---|---|
| 0 | Persiapan, citra uji, jebakan BGR | 10 |
| 1 | Konversi dari definisi: HSI, HSV, Lab, dan skala OpenCV (§6.2) | 25 |
| 2 | Pencahayaan dan derau: apa yang bertahan, apa yang goyah (§6.2, §6.8) | 20 |
| 3 | Pemrosesan citra berwarna: ekualisasi histogram (§6.4&ndash;6.5) | 20 |
| 4 | Segmentasi berbasis warna: RGB, HSV, Lab (§6.5, §6.7) | 30 |
| 5 | Studi kasus: pencarian rakit penolong di perairan | 20 |
| 6 | Laporan ilmiah dan pengumpulan | 10 |


## Bagian 0 &mdash; Persiapan (10 menit)
Jalankan sel berikut **tanpa diubah**.

In [ ]:
# Sel persiapan — jalankan tanpa diubah
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import data, color

plt.rcParams['figure.dpi'] = 110
print('NumPy', np.__version__, '| OpenCV', cv2.__version__)

"""Pembangkit citra uji sintetis Sub-CPMK-4 (benih acak tetap, hanya NumPy).

Semua citra dikembalikan sebagai RGB uint8 (urutan R, G, B). Untuk OpenCV gunakan
cv2.cvtColor(img, cv2.COLOR_RGB2...) atau ubah dulu ke BGR.
Model pencahayaan sengaja disederhanakan: perkalian langsung pada nilai piksel tersandi.
"""
import numpy as np

BENIH = 2027


def _grid(h, w):
    return np.mgrid[0:h, 0:w].astype(np.float64)


def _ke_uint8(x):
    return np.clip(np.floor(255 * x + 0.5), 0, 255).astype(np.uint8)


def _cakram(yy, xx, cy, cx, r):
    return (yy - cy) ** 2 + (xx - cx) ** 2 <= r ** 2


def _kotak(yy, xx, y0, x0, y1, x1):
    return (yy >= y0) & (yy < y1) & (xx >= x0) & (xx < x1)


# ---------------------------------------------------------------- adegan objek berwarna
WARNA_OBJEK = {
    "merah_tua":  (0.78, 0.10, 0.20),   # hue ≈ 351° (sisi magenta dari merah)
    "merah_bata": (0.80, 0.22, 0.10),   # hue ≈ 10°
    "jingga":     (0.92, 0.50, 0.10),
    "kuning":     (0.90, 0.80, 0.15),
    "hijau":      (0.20, 0.60, 0.25),
    "biru":       (0.15, 0.30, 0.75),
    "cokelat":    (0.45, 0.28, 0.16),
    "merah_muda": (0.85, 0.62, 0.66),
}


def objek_warna(rng, h=240, w=360, gelap_kanan=0.35, sigma=0.010):
    """Delapan jenis objek di atas latar netral, pencahayaan menurun linear ke kanan.

    Mengembalikan (citra RGB uint8, masker kebenaran objek merah, peta pencahayaan).
    Objek merah: dua di sisi terang (kiri) dan dua di sisi gelap (kanan).
    """
    yy, xx = _grid(h, w)
    img = np.empty((h, w, 3))
    img[...] = (0.62, 0.60, 0.57)
    img += 0.015 * np.sin(yy / 9.0)[..., None] * np.cos(xx / 13.0)[..., None]
    gt = np.zeros((h, w), bool)
    letak = [  # (warna, bentuk, parameter, merah?)
        ("merah_tua", "c", (60, 50, 26), True), ("merah_bata", "k", (130, 25, 180, 85), True),
        ("merah_tua", "c", (60, 300, 26), True), ("merah_bata", "k", (135, 270, 185, 330), True),
        ("jingga", "c", (60, 125, 22), False), ("kuning", "k", (35, 165, 85, 215), False),
        ("hijau", "c", (200, 150, 22), False), ("biru", "k", (120, 120, 170, 165), False),
        ("cokelat", "k", (195, 210, 230, 260), False), ("merah_muda", "c", (60, 245, 20), False),
        ("jingga", "c", (205, 60, 18), False), ("kuning", "c", (205, 310, 18), False),
    ]
    for nama, bentuk, p, merah in letak:
        m = _cakram(yy, xx, *p) if bentuk == "c" else _kotak(yy, xx, *p)
        img[m] = WARNA_OBJEK[nama]
        if merah:
            gt |= m
    cahaya = 1.0 - (1.0 - gelap_kanan) * xx / (w - 1)
    img = img * cahaya[..., None] + rng.normal(0, sigma, img.shape)
    return _ke_uint8(img), gt, cahaya


# ---------------------------------------------------------------- adegan perairan (SAR)
RAKIT = (0.95, 0.45, 0.08)        # jingga keselamatan (sekoci/rompi)
KAYU = (0.42, 0.27, 0.15)         # puing kayu: corak dekat jingga, gelap, kurang pekat
LAMBUNG = (0.55, 0.12, 0.10)      # cat lambung bawah air (merah tua)
LAUT = (0.07, 0.27, 0.36)


def _geometri_perairan(rng, h, w):
    """Geometri tetap (gelombang, kilau, objek) yang sama untuk semua kondisi cahaya."""
    yy, xx = _grid(h, w)
    gel = (0.55 * np.sin(2 * np.pi * (yy / 23.0 + xx / 61.0)) + 0.35 * np.sin(2 * np.pi * (yy / 11.0 - xx / 37.0))
           + 0.25 * np.sin(2 * np.pi * (xx / 17.0 + yy / 53.0)))
    tekstur = 1.0 + 0.10 * gel / 1.15 + rng.normal(0, 0.03, (h, w))
    # kilau matahari: bercak terang di kanan atas, dekat puncak gelombang
    pusat_kilau = np.exp(-(((yy - 70) / 55.0) ** 2 + ((xx - 310) / 70.0) ** 2))
    kilau = (gel > 0.55) & (rng.random((h, w)) < 0.9 * pusat_kilau)
    obj = {}
    rakit = [(150, 90, 11), (95, 205, 9), (225, 270, 12), (255, 120, 8), (60, 330, 9)]
    rompi = [(175, 330, 4, 6), (205, 180, 4, 6), (120, 55, 4, 5), (40, 140, 4, 6)]
    m_rakit = np.zeros((h, w), bool)
    for cy, cx, r in rakit:
        m_rakit |= _cakram(yy, xx, cy, cx, r)
    for cy, cx, a, b in rompi:
        m_rakit |= ((yy - cy) / a) ** 2 + ((xx - cx) / b) ** 2 <= 1
    m_kayu = np.zeros((h, w), bool)
    for cy, cx, dl, dw, t in [(110, 140, 22, 4, 30), (240, 200, 26, 5, -20), (60, 250, 18, 4, 70), (270, 330, 20, 4, 10)]:
        tr = np.deg2rad(t)
        u, v = (xx - cx) * np.cos(tr) + (yy - cy) * np.sin(tr), -(xx - cx) * np.sin(tr) + (yy - cy) * np.cos(tr)
        m_kayu |= (np.abs(u) <= dl) & (np.abs(v) <= dw)
    m_lambung = ((yy - 40) / 16.0) ** 2 + ((xx - 40) / 34.0) ** 2 <= 1
    obj.update(rakit=m_rakit & ~m_lambung, kayu=m_kayu & ~m_rakit, lambung=m_lambung, kilau=kilau & ~m_rakit & ~m_kayu & ~m_lambung)
    rona = 0.88 + 0.12 * np.cos(2 * np.pi * (yy - xx) / 9.0)        # bayangan lipatan di permukaan objek
    return yy, xx, tekstur, obj, rona


def perairan(rng, kondisi="siang", h=300, w=400, sigma=0.010):
    """Citra udara permukaan laut pascakecelakaan: rakit/rompi jingga, puing kayu, lambung terbalik, kilau.

    kondisi: 'siang' (cahaya putih merata), 'berawan' (redup 0,75 + bayangan awan 0,45),
             'sore' (cahaya hangat: kanal dikali (1,00; 0,82; 0,60) × 0,85).
    Geometri sama untuk semua kondisi (benih geometri tetap); derau berbeda per kondisi.
    Mengembalikan (citra RGB uint8, masker kebenaran rakit/rompi, kamus masker lain).
    """
    yy, xx, tekstur, obj, rona = _geometri_perairan(np.random.default_rng(BENIH + 1), h, w)
    img = np.empty((h, w, 3))
    img[...] = LAUT
    img *= tekstur[..., None]
    img[obj["kilau"]] = 0.80 + 0.15 * rng.random((obj["kilau"].sum(), 1))
    for nama, warna in (("lambung", LAMBUNG), ("kayu", KAYU), ("rakit", RAKIT)):
        m = obj[nama]
        img[m] = np.array(warna) * rona[m][:, None]
    if kondisi == "siang":
        cahaya = np.ones((h, w, 3))
    elif kondisi == "berawan":
        awan = np.exp(-(((yy - 190) / 70.0) ** 2 + ((xx - 150) / 110.0) ** 2) ** 1.5)
        cahaya = np.repeat((0.75 * (1 - 0.40 * awan / awan.max()))[..., None], 3, axis=2)
    elif kondisi == "sore":
        cahaya = np.ones((h, w, 3)) * np.array([1.00, 0.82, 0.60]) * 0.85
    else:
        raise ValueError(kondisi)
    img = img * cahaya + rng.normal(0, sigma, img.shape)
    return _ke_uint8(img), obj["rakit"], obj


def bangkitkan_semua():
    rng = np.random.default_rng(BENIH)
    objek, gt_merah, _ = objek_warna(rng)
    hasil = {"objek": objek, "gt_merah": gt_merah}
    for k in ("siang", "berawan", "sore"):
        im, gt, obj = perairan(rng, k)
        hasil["laut_" + k] = im
    hasil["gt_rakit"] = gt
    hasil["m_kayu"], hasil["m_lambung"], hasil["m_kilau"] = obj["kayu"], obj["lambung"], obj["kilau"]
    return hasil

def tampil(daftar, judul=None, kolom=None, tinggi=3.0, cmap=None):
    n = len(daftar); kolom = kolom or n; baris = int(np.ceil(n / kolom))
    fig, ax = plt.subplots(baris, kolom, figsize=(3.6 * kolom, tinggi * baris)); ax = np.atleast_1d(ax).ravel()
    for a in ax: a.axis('off')
    for a, im, t in zip(ax, daftar, judul or [''] * n):
        a.imshow(im, cmap=cmap if np.ndim(im) == 2 else None); a.set_title(t, fontsize=9)
    plt.tight_layout(); plt.show()

def beda_hue(h1, h2):
    d = np.abs(np.asarray(h1, float) - np.asarray(h2, float)) % 360.0
    return np.minimum(d, 360.0 - d)

def rerata_hue(h):
    t = np.radians(h); return float(np.degrees(np.arctan2(np.sin(t).mean(), np.cos(t).mean())) % 360)

def std_melingkar(h):
    t = np.radians(h); R = np.hypot(np.cos(t).mean(), np.sin(t).mean())
    return float(np.degrees(np.sqrt(-2 * np.log(max(R, 1e-12)))))

def nilai(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool); tp = (pred & gt).sum()
    k = cv2.connectedComponents(pred.astype(np.uint8), connectivity=8)[0] - 1
    return dict(iou=round(float(tp / max((pred | gt).sum(), 1)), 3), presisi=round(float(tp / max(pred.sum(), 1)), 3),
                recall=round(float(tp / max(gt.sum(), 1)), 3), komponen=int(k))

def psnr(a, b):
    mse = np.mean((a.astype(float) - b.astype(float)) ** 2)
    return float('inf') if mse == 0 else float(10 * np.log10(255 ** 2 / mse))

C = bangkitkan_semua()
objek, gt_merah = C['objek'], C['gt_merah']
laut = {k: C['laut_' + k] for k in ('siang', 'berawan', 'sore')}
gt_rakit, m_kayu, m_lambung, m_kilau = C['gt_rakit'], C['m_kayu'], C['m_lambung'], C['m_kilau']
kopi = data.coffee()
YY, XX = np.mgrid[0:240, 0:360]          # koordinat baris, kolom citra objek
tampil([objek, gt_merah, laut['siang'], laut['berawan'], laut['sore'], kopi],
       ['objek (RGB)', 'gt_merah', 'laut siang', 'laut berawan', 'laut sore', 'coffee'], kolom=3, cmap='gray')

In [ ]:
# TODO 0.1 — Jebakan BGR. cv2.imread menghasilkan urutan B, G, R. Tirukan dengan cv2.cvtColor(objek, cv2.COLOR_RGB2BGR),
#            tampilkan hasilnya LANGSUNG dengan plt.imshow, lalu cetak nilai piksel objek[60, 50] dan versi BGR-nya.
bgr = None


## Bagian 1 &mdash; Konversi dari definisi (25 menit)

**HSI menurut Gonzalez** (RGB dinormalisasi ke [0, 1]):

$$\theta = \cos^{-1}\left\{\frac{\tfrac12[(R-G)+(R-B)]}{[(R-G)^2+(R-B)(G-B)]^{1/2}}\right\},\quad H = \begin{cases}\theta & B \le G\\ 360-\theta & B > G\end{cases}\quad \text{(6-16, 6-17)}$$

$$S = 1 - \frac{3}{R+G+B}\min(R,G,B) \quad\text{(6-18)},\qquad I = \tfrac13(R+G+B)\quad\text{(6-19)}$$

Kebalikannya dihitung per sektor 120&deg; (Eq. 6-20 s.d. 6-30). Pada sektor RG (0 &le; H &lt; 120&deg;): $B = I(1-S)$, $R = I[1 + S\cos H/\cos(60^\circ - H)]$, $G = 3I - (R+B)$.
Sektor GB dan BR memakai rumus yang sama setelah H dikurangi 120&deg; atau 240&deg;, dengan peran R, G, B digeser.

**HSV (Smith, 1978; dipakai OpenCV)**: $V = \max(R,G,B)$, $S = (V - \min)/V$, H dihitung per sektor 60&deg;. Pada citra 8-bit OpenCV menyimpan **H/2** (0&ndash;179), S&times;255, V&times;255.

In [ ]:
# TODO 1.1 — Implementasikan rgb_ke_hsi dari Eq. 6-16 s.d. 6-19. Masukan uint8 atau float [0, 1]; keluaran H (derajat), S, I.
#            Tambahkan bilangan kecil pada penyebut (catatan kaki Gonzalez) dan set H = 0 bila S = 0.
def rgb_ke_hsi(rgb):
    ...

H, S, I = rgb_ke_hsi(np.array([[[0.8, 0.5, 0.2]]]))
print('HSI (0.8, 0.5, 0.2):', H[0, 0], S[0, 0], I[0, 0])   # bandingkan dengan hitung tangan di kelas


In [ ]:
# TODO 1.2 — Implementasikan hsi_ke_rgb (Eq. 6-20 s.d. 6-30) untuk ketiga sektor.
#            Uji: konversi `objek` ke HSI lalu kembali; laporkan selisih maksimum (skala 0..1).
def hsi_ke_rgb(H, S, I):
    ...


In [ ]:
# TODO 1.3 — Implementasikan rgb_ke_hsv (heksakerucut). Bandingkan dengan cv2.cvtColor(objek, cv2.COLOR_RGB2HSV):
#            ingat OpenCV 8-bit menyimpan H/2 dan S×255. Laporkan selisih hue maksimum (derajat) dan penyebabnya.
#            Lalu cetak piksel (204, 128, 51) dalam tiga bentuk: HSI Anda, HSV Anda, dan HSV 8-bit OpenCV.
def rgb_ke_hsv(rgb):
    ...


In [ ]:
# TODO 1.4 — Lab. (a) Hitung cv2.cvtColor(p, cv2.COLOR_RGB2Lab) untuk p = (204, 128, 51) dan kembalikan ke L*, a*, b* sebenarnya
#            (OpenCV 8-bit: L×255/100, a+128, b+128). (b) Bandingkan dengan skimage.color.rgb2lab(p).
#            (c) Mengapa jarak Euclid pada keluaran 8-bit OpenCV BUKAN ΔE*ab? Tunjukkan dengan dua warna yang hanya berbeda L*.


In [ ]:
# [Pengayaan] TODO 1.5 — Implementasikan Lab dari definisi (Eq. 6-31 s.d. 6-34) dengan linearisasi sRGB dan matriks sRGB→XYZ (D65).
#            Bandingkan dengan skimage; lalu ulangi TANPA linearisasi. Berapa selisih maksimumnya? Apa artinya bagi rumus di dokumentasi?


**Analisis 1.** (a) Tuliskan nilai HSI, HSV, dan HSV 8-bit OpenCV untuk piksel (204, 128, 51). Mengapa S berbeda antara HSI dan HSV padahal H sama? (b) Dari mana selisih hue &asymp;1&deg; antara implementasi Anda dan OpenCV? (c) Apa akibatnya bila ambang Lab dirancang dari literatur (L* 0&ndash;100) tetapi diterapkan pada keluaran 8-bit OpenCV?

*Jawaban:* ...

## Bagian 2 &mdash; Pencahayaan dan derau (20 menit)

Citra `objek` memuat dua cakram **berwarna sama** (merah tua): satu di sisi terang (pusat baris 60, kolom 50; faktor cahaya 0,91) dan satu di sisi gelap (kolom 300; faktor 0,46).

In [ ]:
# TODO 2.1 — Untuk cakram kiri dan kanan (jari-jari 20 piksel), hitung rerata R, G, B; H, S, V (HSV Anda); H, S, I (HSI);
#            L*, a*, b*, kroma C* = sqrt(a*² + b*²), dan sudut hue h_ab = atan2(b*, a*). Pakai rerata_hue untuk sudut.
#            Susun tabel: kiri, kanan, dan nisbah kanan/kiri (untuk sudut: selisih derajat).
kiri = (YY - 60) ** 2 + (XX - 50) ** 2 <= 20 ** 2
kanan = (YY - 60) ** 2 + (XX - 300) ** 2 <= 20 ** 2


In [ ]:
# TODO 2.2 — Kapan hue tidak bisa dipercaya? Bangkitkan 20000 piksel ber-hue 20° dengan S dan V tertentu,
#            tambahkan derau Gaussian σ = 3 aras per kanal, hitung simpangan baku melingkar hue (std_melingkar).
#            Lakukan untuk S ∈ {0.05, 0.1, 0.2, 0.4, 0.8} (V = 0.8), lalu V ∈ {0.05, 0.1, 0.2, 0.4, 0.8} (S = 0.8).
rng = np.random.default_rng(7)      # jangan diubah agar angka sebanding
def piksel_uji(sat, val, n=20000, sigma=3.0):
    dasar = np.array([1.0, 1 - sat * (1 - 20 / 60), 1 - sat]) * val
    return np.clip(np.floor(dasar * 255 + rng.normal(0, sigma, (n, 3)) + 0.5), 0, 255).astype(np.uint8)[None]


**Analisis 2.** (a) Besaran mana yang **tetap** (nisbah &asymp; 1 atau selisih &asymp; 0&deg;) dan mana yang ikut turun &asymp;50 %? Apakah a* dan b* termasuk yang tetap? (b) Pada S atau V berapa simpangan baku hue melebihi 10&deg;? Mengapa S dan V memberi angka hampir sama (petunjuk: penyebut rumus hue adalah maks &minus; min)? (c) Rumuskan syarat praktis sebelum Anda memakai ambang hue.

*Jawaban:* ...

## Bagian 3 &mdash; Pemrosesan citra berwarna (20 menit)

Gonzalez (§6.5) memperingatkan bahwa ekualisasi histogram setiap kanal RGB secara terpisah menghasilkan warna keliru; yang masuk akal adalah meratakan **intensitas** dan membiarkan hue. Kita ukur.

In [ ]:
# TODO 3.1 — Buat citra redup: redup = 0.85 · (kopi/255)^1.8 (bulatkan ke uint8). Lakukan ekualisasi histogram dengan empat cara:
#   (a) setiap kanal RGB terpisah; (b) kanal I (HSI Anda), kembalikan dengan hsi_ke_rgb lalu clip;
#   (c) kanal V dari cv2 HSV; (d) kanal L* dari Lab (pakai skimage rgb2lab/lab2rgb, L* diskala ke 0..255 untuk equalizeHist).
# Ukur terhadap `redup`: rerata pergeseran hue (beda_hue, hanya piksel S > 0,2 dan V > 0,08), rerata I, rerata S,
# dan untuk (b) persentase piksel yang keluar dari [0, 1] sebelum clip.
redup = np.clip(np.floor(255 * 0.85 * (kopi / 255.0) ** 1.8 + 0.5), 0, 255).astype(np.uint8)


In [ ]:
# [Pengayaan] TODO 3.2 — Subsampling krominans. Ubah kopi ke YCbCr (cv2.COLOR_RGB2YCrCb — perhatikan urutan Y, Cr, Cb!),
#            perkecil Cr dan Cb 2× dan 4× (cv2.resize INTER_AREA, lalu perbesar INTER_LINEAR), kembalikan ke RGB, hitung PSNR.
#            Bandingkan dengan memperkecil R, G, B sama besar sehingga jumlah data sama (faktor per sumbu √(3/(1 + 2/f²))).


**Analisis 3.** (a) Cara mana yang paling merusak hue, dan mengapa? (b) Mengapa ekualisasi kanal I menghasilkan banyak piksel di luar gamut, dan apa akibat pemotongan (clip) terhadap hue? (c) Ekualisasi V menjaga hue hampir sempurna tetapi kecerahannya naik paling sedikit. Jelaskan dari definisi V. (d) Tidak ada cara yang menang di semua ukuran. Mana yang Anda pilih untuk citra kamera kapal yang redup, dan apa harganya?

*Jawaban:* ...

## Bagian 4 &mdash; Segmentasi berbasis warna (30 menit)

Tugas: temukan keempat objek merah pada `objek` (dua merah tua, hue &asymp;351&deg;; dua merah bata, hue &asymp;10&deg;). **Sampel latih hanya boleh diambil dari sisi terang** (kolom &lt; 120), kecuali bila diminta lain. Penilaian memakai `gt_merah` dan fungsi `nilai`.

Jarak Euclid (Eq. 6-48): $D(\mathbf z, \mathbf a) = \lVert \mathbf z - \mathbf a\rVert$. Jarak Mahalanobis (Eq. 6-49): $D(\mathbf z, \mathbf a) = [(\mathbf z - \mathbf a)^T \mathbf C^{-1}(\mathbf z - \mathbf a)]^{1/2}$. Ambang $D_0$ = persentil 99 jarak sampel latih ke reratanya.

In [ ]:
# TODO 4.1 — Aturan RGB tetap dari contoh Blok 1: R > 150, G < 80, B < 80. Nilai dengan nilai(pred, gt_merah).
latih = gt_merah & (XX < 120)
f = objek.reshape(-1, 3).astype(float)
hasil = {}


In [ ]:
# TODO 4.2 — (a) Bola Euclid (Eq. 6-48) dan (b) elipsoid Mahalanobis (Eq. 6-49) dari sampel `latih`.
#            (c) Ulangi Mahalanobis dengan sampel latih ditambah cakram merah tua sisi gelap (pusat 60, 300; r = 20).
def euclid(fitur, sampel):
    ...
def mahalanobis(fitur, sampel):
    ...


In [ ]:
# TODO 4.3 — HSV OpenCV 8-bit. Syarat dasar S ≥ 120 dan V ≥ 40. Bandingkan:
#            (a) satu selang H ≤ 10; (b) dua selang H ≤ 10 atau H ≥ 170; (c) dua selang tanpa syarat S.
hv = cv2.cvtColor(objek, cv2.COLOR_RGB2HSV).astype(int)


In [ ]:
# TODO 4.4 — Lab. (a) Bola Euclid pada bidang a*b* saja (L* diabaikan), sampel `latih`.
#            (b) Selang sudut hue h_ab ≤ 45° atau ≥ 345° dengan kroma C* ≥ 25.
#            Lalu cetak seluruh `hasil` sebagai satu tabel dan tampilkan masker terbaik serta terburuk.


**Analisis 4.** (a) Gonzalez (§6.7) menyatakan segmentasi pada vektor RGB *umumnya* lebih baik daripada HSI, sedangkan banyak tutorial menyatakan sebaliknya. Dengan angka TODO 4.2 dan 4.3, tunjukkan kapan masing-masing benar. (b) Mengapa jarak Euclid pada a*b* gagal sama parahnya dengan RGB, padahal Lab "memisahkan kecerahan"? (c) Mengapa HSV dua selang memberi puluhan komponen walau IoU-nya tinggi, dan dari objek mana komponen palsu itu? (d) Sebutkan satu prasyarat agar jumlah komponen dapat dipakai sebagai ukuran.

*Jawaban:* ...

## Bagian 5 &mdash; Studi kasus: pencarian rakit penolong di perairan (20 menit)

Citra udara sintetis pascakecelakaan kapal memuat **rakit dan rompi jingga** (sasaran), **puing kayu** (hue hampir sama dengan jingga), **lambung merah** kapal terbalik, dan **kilau matahari**. Tiga kondisi cahaya: `siang` (acuan), `berawan` (redup + bayangan awan), `sore` (cahaya hangat). Anda hanya boleh menentukan ambang/sampel dari citra **siang**.

In [ ]:
# TODO 5.1 — Pada laut['siang'], cetak median HSV OpenCV untuk rakit (gt_rakit), kayu (m_kayu), dan lambung (m_lambung).
#            Kanal mana yang memisahkan rakit dari kayu? Tentukan aturan HSV Anda (selang H, ambang S, ambang V).


In [ ]:
# TODO 5.2 — Latih RGB Euclid dan Lab Euclid a*b* dari dua rakit di citra siang (pusat (150, 90) r 11 dan (225, 270) r 12).
#            Terapkan RGB Euclid, Lab a*b*, dan aturan HSV Anda pada ketiga kondisi. Laporkan IoU dan piksel kayu yang ikut terdeteksi.
Y8, X8 = np.mgrid[0:300, 0:400]
latih_r = gt_rakit & (((Y8 - 150) ** 2 + (X8 - 90) ** 2 <= 11 ** 2) | ((Y8 - 225) ** 2 + (X8 - 270) ** 2 <= 12 ** 2))


In [ ]:
# TODO 5.3 — Koreksi warna cahaya. Kilau adalah pantulan cermin sumber cahaya, jadi warnanya menaksir warna cahaya.
#            Ambil 0,5 % piksel paling terang (jumlah R+G+B) pada laut['sore'], hitung rerata warnanya w = (wR, wG, wB),
#            kalikan tiap kanal dengan max(w)/w_i (transformasi per komponen, Eq. 6-38), lalu jalankan ulang TODO 5.2.


**Analisis 5.** (a) Mengapa RGB Euclid dan Lab a*b* yang hampir sempurna di siang runtuh di berawan, sedangkan HSV tidak? (b) Cahaya sore menggagalkan HSV juga. Besaran mana yang bergeser (lihat median TODO 5.1) dan mengapa ambang S = 200 menjadi rapuh? (c) Taksiran warna cahaya dari kilau ternyata sangat tepat. Sebutkan **dua** alasan mengapa hasil ini tidak akan seakurat itu pada foto udara sungguhan. (d) Usulkan urutan pengolahan untuk tim SAR, mulai dari citra mentah sampai masker kandidat rakit.

*Jawaban:* ...

## Bagian 6 &mdash; Laporan ilmiah dan pengumpulan (10 menit)

Indikator kedua Sub-CPMK-4 menilai **laporan praktikum sesuai kaidah penulisan ilmiah**. Laporan Bagian 4&ndash;5 disusun maksimal 4 halaman dengan struktur:

1. **Pendahuluan** &mdash; persoalan (deteksi objek berdasarkan warna di bawah cahaya berubah) dan tujuan percobaan.
2. **Metode** &mdash; ruang warna dan aturan yang diuji, **rumus beserta nomor persamaan Gonzalez**, cara memilih ambang, ukuran evaluasi (IoU, presisi, recall).
3. **Hasil** &mdash; satu tabel bernomor dan berjudul (metode &times; kondisi), satu gambar bernomor dengan keterangan. Angka **disalin dari keluaran notebook**, bukan dibulatkan sesuka hati.
4. **Pembahasan** &mdash; justifikasi pilihan ruang warna **dengan angka**; minimal satu paragraf **batas keberlakuan** (kapan metode Anda gagal dan mengapa).
5. **Kesimpulan** &mdash; menjawab tujuan, tanpa klaim baru.
6. **Daftar rujukan** &mdash; gaya Harvard; minimal Gonzalez dan Woods (2018).

Periksa sebelum mengumpulkan:
- [ ] Semua sel berjalan tanpa galat dari atas (Kernel &rarr; Restart & Run All).
- [ ] Setiap blok Analisis memuat angka dari keluaran Anda.
- [ ] Tabel dan gambar di laporan bernomor, berjudul, dan dirujuk di teks.
- [ ] Tidak ada klaim "HSV selalu lebih baik" tanpa syarat.